## Important: Budget Protection First!

AWS charges for resources you use. So always set up cost alerts BEFORE deploying anything.


# **01. Understanding AWS Services We Use in general**



### AWS Lambda
**Lambda** is AWS's serverless compute service. It runs our code (or a container image) only when it's invoked, and we pay only for the compute time use, billed in millisecond increments. Cold starts (the very first invocation) take a few seconds; warm invocations are fast. Lambda supports container images up to 10GB.

### Lambda Function URLs
**Function URLs** are dedicated HTTPS endpoints that route directly to a Lambda function, so no API Gateway needed, no extra cost. The URL looks like `https://<id>.lambda-url.<region>.on.aws/`. As of November 2025, Function URLs support response streaming, which is what we use to stream Server-Sent Events from FastAPI.

### AWS Lambda Web Adapter
The **Lambda Web Adapter** is an open-source Lambda extension from AWS Labs. By dropping a single binary into your container at `/opt/extensions/lambda-adapter`, you can run any standard web framework (FastAPI, Flask, Express, etc.) on Lambda without modifying your application code. 

### Amazon ECR (Elastic Container Registry)
**ECR** is like GitHub but for Docker images. It's where we store our containerized application before deploying it to Lambda.

### AWS IAM (Identity and Access Management)
**IAM** controls who can access what in your AWS account. We create a special user account with limited permissions for safety, never use your root account for daily work.

### CloudWatch
**CloudWatch** is AWS's monitoring service. It collects logs from your Lambda function and helps you debug issues, like having the browser console for your server.


# **02. Create AWS Account**



### Step 1: Sign Up for AWS

1. Visit [aws.amazon.com](https://aws.amazon.com)
2. Click **Create an AWS Account**
3. Enter your email and choose a password
4. Select **Personal** account type, but not "free tier" - see note below
5. Enter payment information 
6. Verify your phone number via SMS
7. Select **Basic Support - Free**

You now have an AWS root account. This is like having admin access — powerful but dangerous!

> There's an option for first time users of AWS to select the "free tier" of AWS. Don't choose this! It only has limited access to AWS services. It would work for today's projects, but it might not support future projects. This doesn't mean that you need to pay a subscription or pay for support; just that you need to enter payment details and not be in a sandbox environment.


### Step 2: Secure Your Root Account

1. Sign in to AWS Console
2. Click your account name (top right) → **Security credentials**
3. Enable **Multi-Factor Authentication (MFA)**:
   - Click **Assign MFA device**
   - Name: `root-mfa`
   - Select **Authenticator app**
   - Scan QR code with Google Authenticator or Authy
   - Enter two consecutive codes
   - Click **Add MFA**


### Step 3: Set Up Budget Alerts (Critical!)

1. In AWS Console, search for **Billing** and click **Billing and Cost Management**
2. In the left menu, click **Budgets**
3. Click **Create budget**
4. Select **Use a template (simplified)**
5. Choose **Monthly cost budget**
6. Set up three budgets:

**Budget 1 - Early Warning ($1)**:
- Budget name: `early-warning`
- Enter budgeted amount: `1` USD
- Email recipients: Enter your email address
- Click **Create budget**

**Budget 2 - Caution ($5)**:
- Repeat: Create budget → Use a template → Monthly cost budget
- Budget name: `caution-budget`
- Enter budgeted amount: `5` USD
- Email recipients: Enter your email address
- Click **Create budget**

**Budget 3 - Stop Alert ($10)**:
- Repeat: Create budget → Use a template → Monthly cost budget
- Budget name: `stop-budget`
- Enter budgeted amount: `10` USD
- Email recipients: Enter your email address
- Click **Create budget**

If you hit $10, stop and review what's running.

### Step 4: Create an IAM User for Daily Work

Never use your root account for daily work. Let's create a limited user:

1. Search for **IAM** in the AWS Console
2. Click **Users** → **Create user**
3. Username: `aiengineer`
4. Check **Provide user access to the AWS Management Console**
5. Select **I want to create an IAM user**
6. Choose **Custom password** and set a strong password
7. Uncheck **Users must create a new password at next sign-in**
8. Click **Next**

### Step 5: Create a User Group with Permissions

**This section is changed due to the AWS App Runner changes.** The IAM policies attached to the group are different — we replace `AWSAppRunnerFullAccess` with `AWSLambda_FullAccess`. Everything else in this section is identical to the videos.

We'll create a reusable permission group first, then add our user to it:

1. On the permissions page, select **Add user to group**
2. Click **Create group**
3. Group name: `BroadAIEngineerAccess`
4. In the permissions policies search, find and check these policies (note: **the first item replaces what the video shows**):
   - **`AWSLambda_FullAccess`** — to deploy and manage Lambda functions (replaces the video's `AWSAppRunnerFullAccess`)
   - `AmazonEC2ContainerRegistryFullAccess` — to store Docker images
   - `CloudWatchLogsFullAccess` — to view logs
   - `IAMUserChangePassword` — to manage own credentials
   - `IAMFullAccess` — required to let Lambda create its own service-linked execution role
5. Click **Create user group**
6. Back on the permissions page, select the `BroadAIEngineerAccess` group (it should be checked)
7. Click **Next** → **Create user**
8. **Important**: Click **Download .csv file** and save it securely!

It's worth keeping in mind that you might get permissions errors throughout the course, when AWS complains that your user doesn't have permission to do something. The solution is usually to come back to this screen (as the root user) and attach another policy. This is a very common chore working with AWS.

### Step 6: Sign In as IAM User

1. Sign out from root account
2. Go to your AWS sign-in URL (in the CSV file, looks like: `https://123456789012.signin.aws.amazon.com/console`)
3. Sign in with:
   - Username: `aiengineer`
   - Password: (the one you created)

**Checkpoint**: You should see "aiengineer @ Account-ID" in the top right corner.

# **03. Setting Up Docker**

### Step 1: Install Docker Desktop

1. Visit [docker.com/products/docker-desktop](https://www.docker.com/products/docker-desktop)
2. Download for your system:
   - **Mac**: Download for Mac (Apple Silicon or Intel)
   - **Windows**: Download for Windows (requires Windows 10/11)
3. Run the installer
4. **Windows users**: Docker Desktop will install WSL2 if needed — accept all prompts
5. Start Docker Desktop
6. You may need to restart your computer

### Step 2: Verify Docker Works

Open Terminal (Mac) or PowerShell (Windows):

```bash
docker --version
```

You should see: `Docker version 26.x.x` or similar

Test Docker:
```bash
docker run hello-world
```

You should see a message starting with "Hello from Docker!" confirming Docker is working correctly.

**Checkpoint**: Docker Desktop icon should be running (whale icon in system tray/menu bar).
